# 📊 Atomic Step 4: Timeline Refinement & Multi-Format Serialization

This notebook loads the raw diarization timeline JSON, merges consecutive turns of the same speaker separated by small silence gaps (to clean up speech segments), computes speech share analytics, and exports reports in CSV, JSON, RTTM, TXT, and Markdown formats.

In [ ]:
# Install dependencies
!pip install -q pandas
print("[SUCCESS] Dependencies installed!")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.")
except Exception:
    print("[INFO] Already mounted or skipped.")

## ⚙️ Parameters

In [ ]:
# @markdown ### 📁 Timeline Parameters
raw_timeline_json_path = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Outputs/MarauliKhurad1_raw_timeline.json" # @param {type:"string"}
diarization_output_folder = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Atomic Notebooks/Diarization_Outputs/" # @param {type:"string"}

# @markdown ### 📅 Timeline Post-Processing
# @markdown Merge adjacent turns of the same speaker if the silence gap is less than this threshold (seconds):
max_merge_gap = 1.5 # @param {type:"number"}

if not os.path.exists(raw_timeline_json_path):
    print(f"[ERROR] Raw timeline JSON not found at: '{raw_timeline_json_path}'")
else:
    audio_filename = os.path.basename(raw_timeline_json_path)
    audio_name_only = audio_filename.replace("_raw_timeline.json", "").replace(".json", "")
    print(f"[SUCCESS] Validated path. Refined timelines will be saved in: {diarization_output_folder}")

## 📊 Refine Timeline & Export

In [ ]:
import json
import csv
import pandas as pd
from IPython.display import Markdown, display

if os.path.exists(raw_timeline_json_path):
    # 1. Load raw segments
    with open(raw_timeline_json_path, "r", encoding="utf-8") as f:
        raw_speaker_segments = json.load(f)
        
    # 2. Merging helper function
    def merge_speaker_segments(segments, max_gap=1.5):
        if not segments:
            return []
        sorted_segs = sorted(segments, key=lambda x: x["start"])
        merged = []
        current = sorted_segs[0].copy()
        for next_seg in sorted_segs[1:]:
            if next_seg["speaker"] == current["speaker"] and (next_seg["start"] - current["end"]) <= max_gap:
                current["end"] = max(current["end"], next_seg["end"])
            else:
                merged.append(current)
                current = next_seg.copy()
        merged.append(current)
        return merged

    # Process timeline merging
    speaker_segments = merge_speaker_segments(raw_speaker_segments, max_gap=max_merge_gap)

    # 3. Calculate statistics
    speaker_stats = {}
    total_speech_time = 0.0

    for seg in speaker_segments:
        duration = seg["end"] - seg["start"]
        speaker = seg["speaker"]
        total_speech_time += duration
        
        if speaker not in speaker_stats:
            speaker_stats[speaker] = {
                "total_duration": 0.0,
                "turn_count": 0
            }
        speaker_stats[speaker]["total_duration"] += duration
        speaker_stats[speaker]["turn_count"] += 1

    print(f"[SUCCESS] Merged segments down to {len(speaker_segments)} refined speaker turns.")
    
    # Create a specific subfolder for results
    specific_output_folder = os.path.join(diarization_output_folder, audio_name_only)
    os.makedirs(specific_output_folder, exist_ok=True)

    # Define output file paths
    txt_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_diarization_timestamps.txt")
    md_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_diarization_report.md")
    csv_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_timeline.csv")
    json_output_path = os.path.join(specific_output_folder, f"{audio_name_only}_timeline.json")
    rttm_path = os.path.join(specific_output_folder, f"{audio_name_only}_timeline.rttm")

    # Time formatting helper
    def format_time(seconds):
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int((seconds - int(seconds)) * 10)
        if hours > 0:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:01d}"
        else:
            return f"{minutes:02d}:{secs:02d}.{millis:01d}"

    # 1. Export TXT timestamps
    with open(txt_output_path, "w", encoding="utf-8") as txt_file:
        txt_file.write(f"Speaker Diarization Timestamps for: {audio_name_only}\n")
        txt_file.write("=" * 60 + "\n\n")
        for entry in speaker_segments:
            time_str = f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
            txt_file.write(f"{time_str} {entry['speaker']}\n")

    # 2. Export RTTM file
    with open(rttm_path, "w") as rttm_file:
        for entry in speaker_segments:
            duration = entry["end"] - entry["start"]
            # format: SPEAKER <file-id> <channel-id> <start> <duration> <ortho> <look-ahead> <speaker-id> <conf> <key>
            rttm_file.write(f"SPEAKER {audio_name_only.replace(' ', '_')} 1 {entry['start']:.3f} {duration:.3f} <NA> <NA> {entry['speaker']} <NA> <NA>\n")

    # 3. Export JSON refined timeline
    with open(json_output_path, "w", encoding="utf-8") as f:
        json.dump(speaker_segments, f, indent=4, ensure_ascii=False)

    # 4. Export CSV timeline
    with open(csv_output_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Speaker", "Start_Time", "End_Time", "Timestamp_Formatted"])
        for entry in speaker_segments:
            writer.writerow([
                entry["speaker"],
                entry["start"],
                entry["end"],
                f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
            ])

    # 5. Export premium Markdown Report
    with open(md_output_path, "w", encoding="utf-8") as f:
        f.write(f"# 🎙️ Speaker Diarization Analysis Report: {audio_name_only}\n\n")
        f.write("## 📊 Speaker Participation Summary\n\n")
        f.write("| Speaker | Total Speaking Duration | Turn Count | Speech % |\n")
        f.write("| :--- | :--- | :--- | :--- |\n")
        for spk, stats in sorted(speaker_stats.items()):
            percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
            f.write(f"| **{spk}**: | {stats['total_duration']:.2f}s | {stats['turn_count']} | {percentage:.1f}% |\n")

        f.write("\n### 📈 Visual Share of Speech\n\n")
        for spk, stats in sorted(speaker_stats.items()):
            percentage = (stats["total_duration"] / total_speech_time) * 100 if total_speech_time > 0 else 0
            bars = "█" * int(percentage // 4)
            f.write(f"- **{spk}**: `{bars}` ({percentage:.1f}%)\n")

        f.write("\n---\n\n## 📅 Cleaned Timeline\n\n")
        for entry in speaker_segments:
            time_str = f"[{format_time(entry['start'])} - {format_time(entry['end'])}]"
            f.write(f"> **{entry['speaker']}** `{time_str}`\n\n")

    print(f"\n[SUCCESS] Refined timelines and reports successfully serialized inside: '{specific_output_folder}'")